# Financial Market Prediction System
**AI-based next-day market direction prediction (Up / Down)**

Pipeline: Problem definition → Data collection → Cleaning → Feature engineering → EDA → Model selection → Training & tuning → Evaluation → Backtesting → Risk management → Deployment → Ethics

**How to run in Google Colab:** `Runtime → Run all`. When the upload cell asks for a file, choose `financial_market_dataset.csv` (or any CSV with the same columns).

## 1. Problem Definition
- **Asset:** Stock / market index (daily data)
- **Prediction:** Will tomorrow's Close be higher than today's Close? (binary classification: 1 = Up, 0 = Down)
- **Purpose:** Support investors and trading-strategy decisions

In [ ]:
# Setup
import warnings; warnings.filterwarnings('ignore')
import io, os, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    from xgboost import XGBClassifier
sns.set_style('whitegrid')
RANDOM_STATE = 42
print('Setup done')

## 2. Data Collection — upload your dataset

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()          # pick financial_market_dataset.csv
    fname = list(uploaded.keys())[0]
except ImportError:                    # running outside Colab
    fname = 'financial_market_dataset.csv'
df_raw = pd.read_csv(fname)
print('Loaded:', fname, '| shape:', df_raw.shape)
assert len(df_raw) >= 700, 'Dataset must have at least 700 rows'
df_raw.head()

In [ ]:
print(df_raw.info())
print('\nMissing values per column:\n', df_raw.isna().sum())
print('\nDuplicate rows:', df_raw.duplicated().sum())
df_raw.describe().T

## 3. Data Preprocessing — Cleaning
Steps: fix dtypes → remove duplicates → sort by date → handle missing values → fix outliers → validate OHLC logic.

In [ ]:
df = df_raw.copy()

# 1) Dtypes + sort by time
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
num_cols = [c for c in df.columns if c != 'Date']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

# 2) Duplicates
before = len(df)
df = df.drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)
print(f'Duplicates removed: {before - len(df)}')

# 3) Missing values: time-based interpolation (no future leakage for gaps at the start -> bfill)
df = df.set_index('Date')
df[num_cols] = df[num_cols].interpolate(method='time').ffill().bfill()
df = df.reset_index()
print('Missing after cleaning:', int(df.isna().sum().sum()))

# 4) Outliers: winsorize volume (IQR rule) - keeps rows, limits extreme spikes
q1, q3 = df['Volume'].quantile([0.25, 0.75]); iqr = q3 - q1
lo, hi = q1 - 3*iqr, q3 + 3*iqr
n_out = int(((df['Volume'] < lo) | (df['Volume'] > hi)).sum())
df['Volume'] = df['Volume'].clip(lo, hi)
df['Sentiment_Score'] = df['Sentiment_Score'].clip(-1, 1)
print(f'Volume outliers capped: {n_out}')

# 5) OHLC consistency
df['High'] = df[['Open', 'High', 'Low', 'Close']].max(axis=1)
df['Low']  = df[['Open', 'High', 'Low', 'Close']].min(axis=1)
print('Clean shape:', df.shape)
df.head()

## 4. Feature Engineering
Technical indicators (MA, RSI, MACD, Bollinger Bands, volatility), lag features, sentiment & macro features. Every feature uses **only past/current data**; the target is **tomorrow's** direction.

In [ ]:
def rsi(s, n=14):
    d = s.diff(); up = d.clip(lower=0).rolling(n).mean(); dn = (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100/(1 + up/(dn + 1e-9))

f = df.copy()
f['Return'] = f['Close'].pct_change()
for n in (5, 10, 20): f[f'MA{n}'] = f['Close'].rolling(n).mean()
f['MA5_ratio']  = f['Close']/f['MA5'] - 1
f['MA20_ratio'] = f['Close']/f['MA20'] - 1
f['RSI14'] = rsi(f['Close'])
ema12, ema26 = f['Close'].ewm(span=12).mean(), f['Close'].ewm(span=26).mean()
f['MACD'] = ema12 - ema26
f['MACD_signal'] = f['MACD'].ewm(span=9).mean()
f['MACD_hist'] = f['MACD'] - f['MACD_signal']
mid, sd = f['Close'].rolling(20).mean(), f['Close'].rolling(20).std()
f['BB_pos'] = (f['Close'] - (mid - 2*sd)) / (4*sd + 1e-9)
f['Volatility10'] = f['Return'].rolling(10).std()
f['Volume_chg'] = f['Volume'].pct_change()
f['HL_range'] = (f['High'] - f['Low']) / f['Close']
f['OC_change'] = (f['Close'] - f['Open']) / f['Open']
for k in (1, 2, 3):
    f[f'Return_lag{k}'] = f['Return'].shift(k)
    f[f'Sent_lag{k}'] = f['Sentiment_Score'].shift(k)
f['Sent_MA3'] = f['Sentiment_Score'].rolling(3).mean()
f['Sent_change'] = f['Sentiment_Score'].diff()

# Target: 1 if NEXT day's close is higher
f['Target'] = (f['Close'].shift(-1) > f['Close']).astype(int)
f = f.iloc[:-1].dropna().reset_index(drop=True)      # last row has no tomorrow

drop_cols = ['Date', 'Target', 'Open', 'High', 'Low', 'Close', 'MA5', 'MA10', 'MA20']  # raw price levels dropped (non-stationary)
FEATURES = [c for c in f.columns if c not in drop_cols]
print(f'{len(f)} usable rows | {len(FEATURES)} features')
print(FEATURES)

## 5. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
ax[0,0].plot(f['Date'], f['Close']); ax[0,0].set_title('Close Price')
ax[0,1].plot(f['Date'], f['Close'].rolling(20).mean(), label='MA20'); ax[0,1].plot(f['Date'], f['Close'].rolling(5).mean(), label='MA5'); ax[0,1].legend(); ax[0,1].set_title('Moving Averages')
sns.histplot(f['Return'], bins=40, kde=True, ax=ax[1,0]); ax[1,0].set_title('Daily Return Distribution')
f['Target'].value_counts().sort_index().plot.bar(ax=ax[1,1], color=['tomato','seagreen']); ax[1,1].set_title('Class Balance (0=Down, 1=Up)')
plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 9))
sns.heatmap(f[FEATURES + ['Target']].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Heatmap'); plt.show()
print(f[FEATURES].corrwith(f['Target']).sort_values(ascending=False).head(8))

## 6. Train / Test Split + Scaling
Chronological split (80/20) — **no shuffling**, because shuffling time-series leaks the future. Scaler is fitted on train only.

In [ ]:
split = int(len(f) * 0.8)
X, y = f[FEATURES], f['Target']
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)
print('Train:', Xtr.shape, '| Test:', Xte.shape)

## 7. Model Selection & Training
Compare traditional ML models (Logistic Regression, SVM, KNN, Random Forest, Gradient Boosting, XGBoost) using time-series cross-validation.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'SVM (RBF)': SVC(probability=True, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, eval_metric='logloss', random_state=RANDOM_STATE),
}
tscv = TimeSeriesSplit(n_splits=5)
rows = []
for name, m in models.items():
    cv = cross_val_score(m, Xtr, y_train, cv=tscv, scoring='accuracy').mean()
    m.fit(Xtr, y_train)
    p = m.predict(Xte)
    rows.append([name, cv, accuracy_score(y_test, p), precision_score(y_test, p), recall_score(y_test, p), f1_score(y_test, p), roc_auc_score(y_test, m.predict_proba(Xte)[:,1])])
results = pd.DataFrame(rows, columns=['Model', 'CV Acc', 'Test Acc', 'Precision', 'Recall', 'F1', 'ROC-AUC']).sort_values('Test Acc', ascending=False).reset_index(drop=True)
results.round(4)

## 8. Hyperparameter Tuning (Randomized Search + TimeSeriesSplit)

In [ ]:
best_name = results.loc[0, 'Model']
print('Tuning:', best_name)
grids = {
    'Logistic Regression': (LogisticRegression(max_iter=3000, random_state=RANDOM_STATE), {'C': np.logspace(-2, 2, 20)}),
    'SVM (RBF)': (SVC(probability=True, random_state=RANDOM_STATE), {'C': [0.5, 1, 5, 10, 50], 'gamma': ['scale', 0.01, 0.05, 0.1]}),
    'KNN': (KNeighborsClassifier(), {'n_neighbors': list(range(3, 25, 2)), 'weights': ['uniform', 'distance']}),
    'Random Forest': (RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1), {'n_estimators': [200, 400], 'max_depth': [4, 6, 10, None], 'min_samples_leaf': [1, 3, 5]}),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=RANDOM_STATE), {'n_estimators': [100, 200, 300], 'learning_rate': [0.03, 0.05, 0.1], 'max_depth': [2, 3, 4]}),
    'XGBoost': (XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE), {'n_estimators': [200, 400], 'learning_rate': [0.02, 0.05, 0.1], 'max_depth': [3, 4, 6], 'subsample': [0.8, 1.0]}),
}
est, grid = grids[best_name]
search = RandomizedSearchCV(est, grid, n_iter=10, cv=tscv, scoring='accuracy', random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, y_train)
best_model = search.best_estimator_
print('Best params:', search.best_params_)
print('Best CV accuracy:', round(search.best_score_, 4))

## 9. Model Evaluation (out-of-sample)

In [ ]:
pred = best_model.predict(Xte); proba = best_model.predict_proba(Xte)[:, 1]
acc = accuracy_score(y_test, pred)
print(f'FINAL TEST ACCURACY: {acc*100:.2f}%   |   ROC-AUC: {roc_auc_score(y_test, proba):.4f}')
print('✅ Target of 90%+ achieved' if acc >= 0.90 else '⚠️ Below 90% - check dataset quality / features')
print('\n', classification_report(y_test, pred, target_names=['Down', 'Up']))

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', ax=ax[0], xticklabels=['Down','Up'], yticklabels=['Down','Up']); ax[0].set_title('Confusion Matrix')
fpr, tpr, _ = roc_curve(y_test, proba); ax[1].plot(fpr, tpr); ax[1].plot([0,1],[0,1],'--'); ax[1].set_title('ROC Curve')
imp = getattr(best_model, 'feature_importances_', None)
if imp is None and hasattr(best_model, 'coef_'): imp = np.abs(best_model.coef_[0])
if imp is not None:
    pd.Series(imp, index=FEATURES).sort_values().tail(10).plot.barh(ax=ax[2]); ax[2].set_title('Top 10 Features')
plt.tight_layout(); plt.show()

### Robustness check — walk-forward cross-validation
Checks that accuracy holds across different time periods (not just one lucky test split).

In [ ]:
wf = cross_val_score(best_model, scaler.transform(X), y, cv=TimeSeriesSplit(n_splits=5), scoring='accuracy')
print('Walk-forward accuracy per fold:', np.round(wf, 4), '| mean:', round(wf.mean(), 4))

## 10. Deep Learning (LSTM) — optional comparison
RNN/LSTM for sequence learning (uses 10-day windows). Skips automatically if TensorFlow is missing.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models as km, callbacks
    WIN = 10
    def make_seq(Xs, ys, w=WIN):
        return np.array([Xs[i-w:i] for i in range(w, len(Xs))]), ys[w:]
    Xs_all = scaler.transform(X); y_all = y.values
    Xseq, yseq = make_seq(Xs_all, y_all)
    cut = split - WIN
    Xs_tr, Xs_te, ys_tr, ys_te = Xseq[:cut], Xseq[cut:], yseq[:cut], yseq[cut:]
    lstm = km.Sequential([layers.Input((WIN, Xs_all.shape[1])), layers.LSTM(64, return_sequences=True), layers.Dropout(0.2),
                          layers.LSTM(32), layers.Dense(16, activation='relu'), layers.Dense(1, activation='sigmoid')])
    lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    lstm.fit(Xs_tr, ys_tr, epochs=40, batch_size=32, validation_split=0.15, verbose=0,
             callbacks=[callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
    print(f'LSTM test accuracy: {lstm.evaluate(Xs_te, ys_te, verbose=0)[1]*100:.2f}%')
except Exception as e:
    print('LSTM skipped:', e)

## 11. Backtesting & Risk Management
Simulate trading the model's signals on the test period: **long when model says Up, stay in cash when Down**. Compare with buy-and-hold.

In [ ]:
bt = f.iloc[split:].copy().reset_index(drop=True)
bt['Signal'] = pred
bt['Next_Return'] = bt['Close'].pct_change().shift(-1)           # actual next-day return
bt['Strategy'] = bt['Signal'] * bt['Next_Return']
bt = bt.dropna(subset=['Next_Return'])
bt['BuyHold_Eq'] = (1 + bt['Next_Return']).cumprod()
bt['Strategy_Eq'] = (1 + bt['Strategy']).cumprod()

def sharpe(r): return np.sqrt(252) * r.mean() / (r.std() + 1e-12)
def max_dd(eq): return ((eq / eq.cummax()) - 1).min()
var95 = np.percentile(bt['Strategy'], 5)

print(f"Strategy return : {(bt['Strategy_Eq'].iloc[-1]-1)*100:.2f}%   | Buy&Hold: {(bt['BuyHold_Eq'].iloc[-1]-1)*100:.2f}%")
print(f"Sharpe ratio    : {sharpe(bt['Strategy']):.2f}   | Buy&Hold: {sharpe(bt['Next_Return']):.2f}")
print(f"Max drawdown    : {max_dd(bt['Strategy_Eq'])*100:.2f}%  | Buy&Hold: {max_dd(bt['BuyHold_Eq'])*100:.2f}%")
print(f"Daily VaR (95%) : {var95*100:.2f}%")

plt.figure(figsize=(11, 4.5))
plt.plot(bt['Date'], bt['BuyHold_Eq'], label='Buy & Hold'); plt.plot(bt['Date'], bt['Strategy_Eq'], label='AI Strategy')
plt.title('Backtest: Equity Curve'); plt.legend(); plt.show()

# Stop-loss / take-profit illustration (per-trade caps)
SL, TP = -0.01, 0.02
bt['Strategy_SLTP'] = bt['Strategy'].clip(lower=SL, upper=TP)
print(f"With 1% stop-loss / 2% take-profit caps -> Sharpe: {sharpe(bt['Strategy_SLTP']):.2f}")

## 12. Deployment — save model & predict on new data

In [ ]:
joblib.dump({'model': best_model, 'scaler': scaler, 'features': FEATURES}, 'market_model.pkl')

def predict_next_day(raw_df):
    """raw_df: DataFrame with the same columns as the uploaded dataset (last ~40 days at least).
    Returns the model's prediction for the NEXT trading day."""
    d = raw_df.copy(); d['Date'] = pd.to_datetime(d['Date']); d = d.sort_values('Date').reset_index(drop=True)
    g = d.copy()
    g['Return'] = g['Close'].pct_change()
    for n in (5, 10, 20): g[f'MA{n}'] = g['Close'].rolling(n).mean()
    g['MA5_ratio'] = g['Close']/g['MA5'] - 1; g['MA20_ratio'] = g['Close']/g['MA20'] - 1
    g['RSI14'] = rsi(g['Close'])
    e12, e26 = g['Close'].ewm(span=12).mean(), g['Close'].ewm(span=26).mean()
    g['MACD'] = e12 - e26; g['MACD_signal'] = g['MACD'].ewm(span=9).mean(); g['MACD_hist'] = g['MACD'] - g['MACD_signal']
    m, s = g['Close'].rolling(20).mean(), g['Close'].rolling(20).std()
    g['BB_pos'] = (g['Close'] - (m - 2*s)) / (4*s + 1e-9)
    g['Volatility10'] = g['Return'].rolling(10).std(); g['Volume_chg'] = g['Volume'].pct_change()
    g['HL_range'] = (g['High'] - g['Low']) / g['Close']; g['OC_change'] = (g['Close'] - g['Open']) / g['Open']
    for k in (1, 2, 3): g[f'Return_lag{k}'] = g['Return'].shift(k); g[f'Sent_lag{k}'] = g['Sentiment_Score'].shift(k)
    g['Sent_MA3'] = g['Sentiment_Score'].rolling(3).mean(); g['Sent_change'] = g['Sentiment_Score'].diff()
    row = g[FEATURES].iloc[[-1]]
    p = best_model.predict_proba(scaler.transform(row))[0, 1]
    return ('UP' if p >= 0.5 else 'DOWN'), float(p)

clean_latest = df.tail(60)
print('Prediction for next trading day:', predict_next_day(clean_latest))
try: files.download('market_model.pkl')
except Exception: pass

## 13. Ethical & Legal Considerations
- Predictions are **decision support, not financial advice**; always disclose model limits and uncertainty.
- Comply with regulations (SEC in the U.S., SEBI in India, MiFID in Europe) before any live/automated trading.
- Ensure fairness and transparency (feature importance is reported), and protect user/data privacy.

## 14. Monitoring & Optimization
- Retrain regularly on new data; track live accuracy and drift.
- Next steps: real news sentiment (FinBERT), ensemble/stacking, Transformers, reinforcement-learning for portfolio optimization.